### 1-Setup

In [3]:
import os
import numpy as np
import pandas as pd

for d in ("data/raw", "data/own", "data/processed", "figures"):
    os.makedirs(d, exist_ok=True)

### 2-IMDB 50K

In [7]:
import os
import pandas as pd

path = "data/raw/imdb_50k.csv"

if not os.path.exists(path):
    url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
    imdb = pd.read_csv(url)
    imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    imdb.to_csv(path, index=False)

imdb = pd.read_csv(path)

if "label" not in imdb.columns:
    imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
    imdb.to_csv(path, index=False)

print(imdb.shape, imdb["label"].value_counts().to_dict())

(50000, 3) {1: 25000, 0: 25000}


### 3-TMDB

In [1]:
import time, requests
from getpass import getpass

BASE = "https://api.themoviedb.org/3"
KEY = "e597beecc9b84bafbd514237ec8ebde0"
GENRES = {"Action": 28, "Comedy": 35, "Drama": 18, "Horror": 27,
          "Sci-Fi": 878, "Animation": 16, "Romance": 10749, "Thriller": 53}

def get(path, **params):
    for attempt in range(4):
        try:
            r = requests.get(f"{BASE}{path}", params={"api_key": KEY, **params}, timeout=20)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        if r.status_code == 401:
            raise RuntimeError("Invalid TMDB API key (401)")
        if r.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
        time.sleep(0.15)
        return r.json()
    return {"results": []}

def collect(extra, pages=4, start_page=1):
    rows = []
    for genre, gid in GENRES.items():
        for page in range(start_page, start_page + pages):
            res = get("/discover/movie", with_genres=gid, page=page, language="en-US",
                      **{"primary_release_date.gte": "2016-01-01"}, **extra)["results"]
            for m in res:
                data = get(f"/movie/{m['id']}/reviews", language="en-US")
                for rv in data.get("results", []):
                    rating = (rv.get("author_details") or {}).get("rating")
                    if rating is None:
                        continue
                    rows.append({"review": rv["content"], "rating": rating,
                                 "film": m["title"], "year": (m.get("release_date") or "")[:4],
                                 "genre": genre, "source": "TMDB", "review_id": rv["id"]})
        print(genre, len(rows))
    return pd.DataFrame(rows)

### Expand and Accumulate Raw Reviews

In [ ]:
import os
import time
import requests
import pandas as pd


BASE = "https://api.themoviedb.org/3"
KEY = "e597beecc9b84bafbd514237ec8ebde0"
raw_path = "data/own/tmdb_raw.csv"

# Load existing if any, or start fresh
if os.path.exists(raw_path):
    tmdb_all = pd.read_csv(raw_path)
else:
    tmdb_all = pd.DataFrame()

all_rows = []

# Pull pages 1 to 15 to gather a large pool of candidate reviews
for page in range(1, 16):
    url = f"{BASE}/discover/movie"
    params = {
        "api_key": KEY, 
        "language": "en-US", 
        "sort_by": "popularity.desc",
        "primary_release_date.gte": "2016-01-01",
        "page": page
    }
    
    try:
        res = requests.get(url, params=params, timeout=4).json()
    except:
        continue
        
    for m in res.get("results", []):
        movie_id = m["id"]
        title = m["title"]
        year = (m.get("release_date") or "")[:4]
        
        rev_url = f"{BASE}/movie/{movie_id}/reviews"
        try:
            rev_res = requests.get(rev_url, params={"api_key": KEY}, timeout=3).json()
            for rv in rev_res.get("results", []):
                rating = (rv.get("author_details") or {}).get("rating")
                if rating is not None:
                    all_rows.append({
                        "review": rv["content"], 
                        "rating": rating,
                        "film": title, 
                        "year": year,
                        "genre": "Various", 
                        "source": "TMDB", 
                        "review_id": rv["id"]
                    })
        except:
            continue
    print(f"Completed page {page}. Accumulated pool size so far: {len(all_rows)}")
    time.sleep(0.1)

df_new = pd.DataFrame(all_rows)
tmdb_all = pd.concat([tmdb_all, df_new]).drop_duplicates(subset="review_id").reset_index(drop=True)
tmdb_all.to_csv(raw_path, index=False)
print(f"\nTotal unique raw reviews in pool: {len(tmdb_all)}")

Completed page 1. Accumulated pool size so far: 46
Completed page 2. Accumulated pool size so far: 87
Completed page 4. Accumulated pool size so far: 185
Completed page 5. Accumulated pool size so far: 279
Completed page 6. Accumulated pool size so far: 366
Completed page 7. Accumulated pool size so far: 440
Completed page 8. Accumulated pool size so far: 529
Completed page 9. Accumulated pool size so far: 633
Completed page 10. Accumulated pool size so far: 771
Completed page 11. Accumulated pool size so far: 902
Completed page 12. Accumulated pool size so far: 1038
Completed page 13. Accumulated pool size so far: 1128
Completed page 14. Accumulated pool size so far: 1245
Completed page 15. Accumulated pool size so far: 1368

Total unique raw reviews in pool: 1407


### Filter and Produce the Final 300 Balanced Reviews

In [5]:
import os
import pandas as pd

# 1. Automatically find where 'tmdb_raw.csv' is saved
possible_paths = ["data/own/tmdb_raw.csv", "../data/own/tmdb_raw.csv", "tmdb_raw.csv"]
raw_path = None
for p in possible_paths:
    if os.path.exists(p):
        raw_path = p
        break

if raw_path is None:
    raise FileNotFoundError("Could not find 'tmdb_raw.csv'. Please run the fetch cell first!")

print(f"Loading raw reviews from: {raw_path}")
tmdb_all = pd.read_csv(raw_path)
print(f"Total raw reviews available: {len(tmdb_all)}")

# 2. Filter by word length (20 to 300 words)
tmdb_all["word_count"] = tmdb_all["review"].apply(lambda x: len(str(x).split()))
df_filtered = tmdb_all[(tmdb_all["word_count"] >= 20) & (tmdb_all["word_count"] <= 300)].copy()

# 3. Assign labels based on your report rules (>= 7 is positive, <= 4 is negative)
def assign_label(rating):
    if rating >= 7.0:
        return 1
    elif rating <= 4.0:
        return 0
    return None

df_filtered["label"] = df_filtered["rating"].apply(assign_label)
df_filtered = df_filtered.dropna(subset=["label"])
df_filtered["label"] = df_filtered["label"].astype(int)

pos_reviews = df_filtered[df_filtered["label"] == 1]
neg_reviews = df_filtered[df_filtered["label"] == 0]

print(f"Filtered pool -> Positive: {len(pos_reviews)} | Negative: {len(neg_reviews)}")

# 4. Take up to 150 of each (or whatever is available if pool is smaller right now)
n_pos = min(150, len(pos_reviews))
n_neg = min(150, len(neg_reviews))

df_final = pd.concat([pos_reviews.head(n_pos), neg_reviews.head(n_neg)]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Add required columns matching your report schema
df_final["hard_case"] = False
df_final = df_final[["review", "label", "rating", "source", "film", "genre", "year", "hard_case"]]

# 5. Ensure target folders exist in both locations to prevent path errors
os.makedirs("data/own", exist_ok=True)
os.makedirs("../data/own", exist_ok=True)

# Save to both locations so your project structure is covered regardless of working directory
df_final.to_csv("data/own/own_reviews.csv", index=False)
df_final.to_csv("data/own/to_annotate.csv", index=False)
df_final.to_csv("../data/own/own_reviews.csv", index=False)
df_final.to_csv("../data/own/to_annotate.csv", index=False)

print(f"\nSuccessfully prepared dataset with {len(df_final)} reviews!")
print(f"Positive: {len(df_final[df_final['label'] == 1])} | Negative: {len(df_final[df_final['label'] == 0])}")
print("Saved to data/own/own_reviews.csv and to_annotate.csv!")

Loading raw reviews from: data/own/tmdb_raw.csv
Total raw reviews available: 1407
Filtered pool -> Positive: 642 | Negative: 146

Successfully prepared dataset with 296 reviews!
Positive: 150 | Negative: 146
Saved to data/own/own_reviews.csv and to_annotate.csv!
